# 01. 미 국채 커브 읽기

재무부 par yield로 UST 할인 커브를 만들고, 커브 모양과 포지션의 key rate DV01을 읽는다.

- 데이터: `data/ust_par/*.csv` (`python pipeline/ust_par.py 2026`으로 수집). 없으면 `tests/fixtures/ust_par_sample.csv` **합성 샘플**로 돌아가며, 그때 숫자는 시장 해석에 쓰지 않는다.
- 흐름: `Snapshot.build` → 헬퍼 재가격 진단 → 커브 표·모양 → 격자 시나리오 시장 → key rate DV01

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from quanttool.analytics.market_monitor import curve_shape, curve_table
from quanttool.analytics.sensitivity import key_rate_dv01, parallel_dv01
from quanttool.config import load_config
from quanttool.derivatives import Portfolio
from quanttool.market import Snapshot, UstParCsvLoader
from quanttool.market.diagnostics import helper_repricing
from quanttool.scenario import ScenarioGrid, ScenarioMarket, ScenarioRunner, SensitivityGenerator

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
files = sorted((ROOT / "data" / "ust_par").glob("*.csv"))
SAMPLE = not files
if SAMPLE:
    files = [ROOT / "tests" / "fixtures" / "ust_par_sample.csv"]
    print("경고: 실제 데이터가 없어 합성 샘플을 쓴다. 아래 숫자는 시장 해석용이 아니다.")
loader = UstParCsvLoader(files)
cfg = load_config(ROOT / "configs")
ASOF = loader.dates()[-1]
snap = Snapshot.build(ASOF, cfg, loader)
ASOF

## 1. 구축 진단
헬퍼 재가격 오차가 0.01bp를 넘으면 커브를 믿지 않는다.

In [ ]:
helper_repricing(snap, "UST").round({"market": 5, "implied": 5, "error_bp": 6})

## 2. 커브 표와 모양
- `par`: 커브로 다시 계산한 par yield (시장 시세와 같아야 함)
- `zero_cc`: 연속복리 제로금리, `fwd_1y`: 해당 만기에서 시작하는 1년 선도금리
- 선도가 제로보다 높은 구간은 시장이 그 구간 금리 상승(또는 기간 프리미엄)을 가격에 반영한 곳이다.

In [ ]:
table = curve_table(snap, "UST", cfg)
(table[["par", "zero_cc", "fwd_1y"]] * 100).round(3)

In [ ]:
ax = (table[["par", "zero_cc", "fwd_1y"]] * 100).set_index(table["years"]).plot(marker="o")
ax.set_xlabel("years")
ax.set_ylabel("%")
ax.set_title(f"UST curve {ASOF}" + (" (synthetic sample)" if SAMPLE else ""))
plt.show()

기울기·곡률 (bp). 2s10s < 0이면 역전, 2s5s10s > 0이면 중기 구간이 양 끝보다 높다(배가 나온 모양).

In [ ]:
curve_shape(table).round(1)

데이터에 여러 날짜가 있으면 모양 지표의 최근 추이를 본다. 날짜마다 스냅샷을 순차로 만든다(전역 평가일).

In [ ]:
history = {}
for d in loader.dates()[-60:]:
    s = Snapshot.build(d, cfg, loader)
    history[d] = curve_shape(curve_table(s, "UST", cfg))
history = pd.DataFrame(history).T
if len(history) > 1:
    history[["2s10s", "5s30s", "2s5s10s"]].plot(title="curve shape (bp)")
    plt.show()
history.tail()

## 3. 포지션 key rate DV01
격자 시나리오 시장(제로금리 격자, 선형 보간)에서 격자점을 하나씩 1bp 올린다. 먼저 격자 시장 기준 NPV가 스냅샷 NPV와 맞는지 대사한다.

In [ ]:
snap = Snapshot.build(ASOF, cfg, loader)
book = Portfolio.load(ROOT / "positions" / "ust_book.yaml")
grid = ScenarioGrid(cfg.grid)
res = ScenarioRunner(book, ScenarioMarket(snap, grid), cfg).run(
    SensitivityGenerator(grid, cfg.sensitivity).generate()
)
res.reconciliation

In [ ]:
krd = key_rate_dv01(res, cfg.sensitivity, "UST")
par = parallel_dv01(res, cfg.sensitivity, "UST")
print(
    "Σ key rate DV01 =", round(krd["total"].sum(), 2), " / parallel DV01 =", round(par["total"], 2)
)
krd.round(0)

In [ ]:
ax = krd.drop(columns="total").plot.bar(stacked=True, figsize=(10, 4))
krd["total"].plot(ax=ax, color="k", marker="o", label="total")
ax.set_ylabel("PnL per 1bp (USD)")
ax.legend()
plt.show()

읽는 법
- 막대의 부호: 롱 채권은 금리 상승에 손실(음수), 숏은 이익.
- `total` 선이 어느 만기에 몰렸는지가 포트폴리오의 실질 듀레이션 위치다. 평행 DV01이 작아도 만기별로 상쇄된 것이면 커브 기울기(2s10s, 5s30s) 위험은 남는다.
- 이 DV01은 **격자 제로금리 기준**이다. 헷지 수량(국채 par 기준)으로 바꾸는 것은 다음 단계 `scenario/par.py`.